# VRAM probe

Runs a few real training iterations and a few predictions for every model x input size x candidate batch size, and records peak GPU memory and speed. The recommendation table at the end gives, per run, the batch size and gradient accumulation to put into `configs/runs/<run_id>.yaml`, and an estimate of the training time against the 2-hour cap.

- Logic: `iiss.models.probe`. Results: `results/vram_probe/` (keep the outputs of this notebook: they are evidence).
- Every probe runs in its own Python subprocess, so a CUDA out-of-memory crash cannot break the next probe. Training peaks above the physical VRAM (Windows shared-memory spill) are flagged as `spill` and never count as fitting.
- The full matrix takes about 20-40 min on the local GPU. Close other GPU programs first.
- `SMOKE = True`: S=320 only, smallest batch, 5 iterations, 16 images; writes to `runs/smoke/`.

## Parameters

In [1]:
MODELS = ['yolo', 'maskrcnn', 'm2f']
SIZES = [320, 640, 960]
BATCHES = {'yolo': [4, 8, 16], 'maskrcnn': [1, 2, 4], 'm2f': [1, 2, 4]}
ITERS = 20            # training iterations per probe
WARMUP = 3            # first iterations not timed
INFER_IMAGES = 10     # valid images for the inference timing
BUDGET_MB = 5600      # max training peak (reserved) that counts as fitting on 6 GB
OUT_DIR = 'results/vram_probe'
FRESH = True          # delete the previous probe.csv first
SMOKE = False

In [2]:
import pandas as pd
import torch
from IPython.display import display

from iiss.models.common import apply_smoke
from iiss.models.probe import probe_matrix, recommend
from iiss.utils.config import load_run_config
from iiss.utils.log import setup_logging
from iiss.utils.paths import repo_root, resolve_path

setup_logging()
ROOT = repo_root()
if SMOKE:
    SIZES, ITERS, WARMUP, INFER_IMAGES = [320], 5, 1, 3
    BATCHES = {m: [min(b)] for m, b in BATCHES.items()}
    OUT_DIR = 'runs/smoke/vram_probe'
out_dir = resolve_path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
if FRESH:
    (out_dir / 'probe.csv').unlink(missing_ok=True)
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

2.14.1+cu126 NVIDIA GeForce RTX 4050 Laptop GPU


## Run configs to probe

In [3]:
configs = []
for model in MODELS:
    for size in SIZES:
        cfg = load_run_config(ROOT / 'configs' / 'runs' / f'{model}_s{size}_seed42.yaml')
        configs.append(apply_smoke(cfg) if SMOKE else cfg)
[c['run_id'] for c in configs]

['yolo_s320_seed42',
 'yolo_s640_seed42',
 'yolo_s960_seed42',
 'maskrcnn_s320_seed42',
 'maskrcnn_s640_seed42',
 'maskrcnn_s960_seed42',
 'm2f_s320_seed42',
 'm2f_s640_seed42',
 'm2f_s960_seed42']

## Probe

One row per model x S x batch. Larger batches are skipped once a smaller one runs out of memory, spills or exceeds `BUDGET_MB`.

In [4]:
results = probe_matrix(configs, BATCHES, out_dir / 'probe.csv', budget_mb=BUDGET_MB,
                       iters=ITERS, warmup=WARMUP, infer_images=INFER_IMAGES)
display(results)

2026-11-23 00:47:18 INFO    iiss.models.probe: Probing yolo S=320 batch=4 (subprocess)


2026-11-23 00:48:42 INFO    iiss.models.probe: -> {'model': 'yolo', 'input_size': 320, 'batch': 4, 'iters_timed': 16, 'oom': False, 'peak_train_mb': 656.0, 's_per_iter': 0.068, 'infer_ms': 48.8, 'peak_infer_mb': 130.0, 'note': '', 'gpu_total_mb': 6140.5, 'spill': False} (84 s)


2026-11-23 00:48:42 INFO    iiss.models.probe: Probing yolo S=320 batch=8 (subprocess)


2026-11-23 00:49:56 INFO    iiss.models.probe: -> {'model': 'yolo', 'input_size': 320, 'batch': 8, 'iters_timed': 16, 'oom': False, 'peak_train_mb': 916.0, 's_per_iter': 0.079, 'infer_ms': 48.5, 'peak_infer_mb': 130.0, 'note': '', 'gpu_total_mb': 6140.5, 'spill': False} (74 s)


2026-11-23 00:49:56 INFO    iiss.models.probe: Probing yolo S=320 batch=16 (subprocess)


2026-11-23 00:51:05 INFO    iiss.models.probe: -> {'model': 'yolo', 'input_size': 320, 'batch': 16, 'iters_timed': 16, 'oom': False, 'peak_train_mb': 1624.0, 's_per_iter': 0.098, 'infer_ms': 44.0, 'peak_infer_mb': 130.0, 'note': '', 'gpu_total_mb': 6140.5, 'spill': False} (69 s)


... (41 outputs left out)

2026-11-23 01:08:46 INFO    iiss.models.probe: -> {'model': 'm2f', 'input_size': 960, 'batch': 1, 'iters_timed': 17, 'oom': False, 'peak_train_mb': 5432.0, 's_per_iter': 0.437, 'infer_ms': 360.6, 'peak_infer_mb': 2592.0, 'note': '', 'gpu_total_mb': 6140.5, 'spill': False} (39 s)


2026-11-23 01:08:46 INFO    iiss.models.probe: Probing m2f S=960 batch=2 (subprocess)


2026-11-23 01:12:30 INFO    iiss.models.probe: -> {'model': 'm2f', 'input_size': 960, 'batch': 2, 'iters_timed': 17, 'oom': False, 'peak_train_mb': 10192.0, 's_per_iter': 9.514, 'infer_ms': 477.7, 'peak_infer_mb': 2640.0, 'note': 'peak above physical VRAM (spill)', 'gpu_total_mb': 6140.5, 'spill': True} (224 s)


,model,input_size,batch,iters_timed,oom,peak_train_mb,s_per_iter,infer_ms,peak_infer_mb,note,gpu_total_mb,spill
0,yolo,320,4,16,False,656.0,0.068,48.8,130.0,,6140.5,False
1,yolo,320,8,16,False,916.0,0.079,48.5,130.0,,6140.5,False
2,yolo,320,16,16,False,1624.0,0.098,44.0,130.0,,6140.5,False
3,yolo,640,4,16,False,2320.0,0.097,61.1,412.0,,6140.5,False
4,yolo,640,8,16,False,3374.0,0.171,56.9,412.0,,6140.5,False
5,yolo,640,16,16,False,5764.0,0.341,56.5,408.0,,6140.5,False
6,yolo,960,4,16,False,5470.0,0.211,218.2,412.0,,6140.5,False
7,yolo,960,8,16,False,7882.0,1.551,195.6,358.0,peak above physical VRAM (spill),6140.5,True
8,yolo,960,16,0,False,NaN,NaN,NaN,NaN,skipped: batch 8 already exceeds the 5600 MB b...,NaN,False
9,maskrcnn,320,1,17,False,1132.0,0.100,166.4,672.0,,6140.5,False


## Recommendation

`batch` keeps the configured effective batch (with `accumulate`); `max_fit_batch` is the largest batch that fits at all. `est_train_h` is a planning estimate for the full epoch budget including validation; `cap_risk` = above 2 h.

In [5]:
rec = recommend(results, {c['run_id']: c for c in configs}, budget_mb=BUDGET_MB)
rec.to_csv(out_dir / 'recommendation.csv', index=False)
display(rec)

,run_id,batch,accumulate,effective_batch,max_fit_batch,peak_train_mb,s_per_iter,infer_ms,est_train_h,cap_risk,note
0,yolo_s320_seed42,16,1,64,16,1624.0,0.098,44.0,0.27,False,
1,yolo_s640_seed42,8,1,64,8,3374.0,0.171,56.9,0.50,False,
2,yolo_s960_seed42,4,1,64,4,5470.0,0.211,218.2,1.59,False,
3,maskrcnn_s320_seed42,2,1,2,4,1860.0,0.168,169.6,0.32,False,
4,maskrcnn_s640_seed42,2,1,2,4,2084.0,0.214,156.3,0.37,False,
5,maskrcnn_s960_seed42,2,1,2,4,2692.0,0.309,317.5,0.59,False,
6,m2f_s320_seed42,2,1,2,4,1924.0,0.225,96.2,0.67,False,
7,m2f_s640_seed42,2,1,2,2,5342.0,0.399,115.7,1.16,False,
8,m2f_s960_seed42,1,2,2,1,5432.0,0.437,360.6,2.58,True,


## Next step

Send `results/vram_probe/recommendation.csv` (or this table) back to Claude Code. Batch overrides then go into `configs/runs/<run_id>.yaml`, and the epoch budgets are confirmed or adjusted per model (identical across S).